# 02 – Text features
Input: `data_clean/reviews_laundry.csv`  
Output: `data_clean/reviews_laundry_features.csv`

Builds attribute flags from a keyword dictionary, then cross-checks with TF-IDF and LDA.

In [1]:
import re
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

d = pd.read_csv("data_clean/reviews_laundry.csv")
d["text_l"] = d.text.str.lower()
d.shape

(3939, 21)

## 1. Attribute dictionary
Refine after reading ~100 reviews: `d.sample(100, random_state=1).text`

In [2]:
ATTR = {
    # product
    "reliability":  r"broke|stopped working|stop working|\bfail(?:ed|s|ure)?\b|died|defect|lemon|\bquit(?:s)?\b|malfunction|\bdead\b|won.?t (?:start|turn on|power on|spin|drain)",
    "noise":        r"\bloud|noise|noisy|vibrat|rattl|grind|squeal|\bbang",
    "cleaning":     r"doesn.?t clean|not clean|smell|odor|mold|mildew|residue|\bstain(?:s|ed|ing)?\b",
    "design_usability": r"\bdoor|\blid\b|display|\bapp\b|wifi|control panel|button|drawer|dispenser",
    "value":        r"price|expensive|overpriced|worth|cheap|money",
    # service
    "repair_service": r"technician|repair|service call|\bparts?\b|warranty|serviced|service tech",
    "support":      r"customer service|customer support|(?:tech|technical|lg|ge|samsung|whirlpool|bosch) support|support (?:team|line|agent)|\brep\b|on hold|phone call|called lg|called them|call center",
    "delivery_install": r"\bdent(?:s|ed)?\b|damaged (?:on|during|in) (?:delivery|shipping|transit)|arrived damaged|scratch|late delivery|missed (?:the )?delivery|no.?show|install(?:er|ers|ation)? (?:was|were)? ?(?:wrong|bad|poor|damaged|broke)|improperly installed|delivery (?:was|were) (?:late|delayed|rescheduled)",
    "return_refund": r"refund|\breturn(?:ed|ing)? (?:it|the|them|this)|exchange|money back|replace(?:ment)? (?:the )?(?:unit|washer|dryer|machine)|new (?:unit|machine) (?:was|were) sent",
}
for k, pat in ATTR.items():
    d[f"a_{k}"] = d.text_l.str.contains(pat, regex=True).astype(int)
d[[f"a_{k}" for k in ATTR]].mean().round(3).sort_values(ascending=False)

a_repair_service      0.366
a_value               0.275
a_reliability         0.238
a_design_usability    0.221
a_cleaning            0.179
a_noise               0.143
a_support             0.107
a_return_refund       0.054
a_delivery_install    0.014
dtype: float64

In [3]:
# Validation: hand-label a sample, then compute precision per attribute.
# 1) export 100 reviews, 2) add a column per attribute with your manual 0/1, 3) compare.
d.sample(100, random_state=1)[["review_id", "text"] + [f"a_{k}" for k in ATTR]].to_csv("data_clean/manual_validation_sample.csv", index=False)

## 2. TF-IDF: terms that separate low from high reviews

In [4]:
m = d.dropna(subset=["low"])
tf = TfidfVectorizer(stop_words="english", min_df=10, ngram_range=(1, 2), max_features=20000)
X = tf.fit_transform(m.text)
terms = np.array(tf.get_feature_names_out())
low_mean = np.asarray(X[m.low.values == 1].mean(axis=0)).ravel()
high_mean = np.asarray(X[m.low.values == 0].mean(axis=0)).ravel()
diff = low_mean - high_mean
print("LOW:", terms[np.argsort(diff)[-25:][::-1]])
print("HIGH:", terms[np.argsort(diff)[:25]])

LOW: ['service' 'buy' 'whirlpool' 'samsung' 'warranty' 'repair' 'worst' 'years'
 'junk' 'lg' 'customer' 'months' 'customer service' 'parts' 'year' 'ge'
 'piece' 'fix' 'money' 'broke' 'called' 'waste' 'horrible' 'said'
 'terrible']
HIGH: ['great' 'love' 'stars' 'works' 'quiet' 'easy' 'works great' 'good' 'best'
 'far' 'dry' 'happy' 'nice' 'dryer' 'washer dryer' 'delivery' 'perfect'
 'excellent' 'set' 'drying' 'awesome' 'space' 'great washer' 'use'
 'little']


## 3. LDA topics (optional cross-check on the dictionary)

In [5]:
cv = CountVectorizer(stop_words="english", min_df=10, max_df=0.5)
Xc = cv.fit_transform(m.text)
lda = LatentDirichletAllocation(n_components=6, random_state=0).fit(Xc)
w = np.array(cv.get_feature_names_out())
for i, t in enumerate(lda.components_):
    print(i, w[t.argsort()[-12:][::-1]])

0 ['machine' 'clothes' 'load' 'water' 'cycle' 'washing' 'just' 'buy' 'spin'
 'time' 'wash' 'don']
1 ['dryer' 'service' 'machine' 'samsung' 'whirlpool' 'product' 'new' 'great'
 'lg' 'customer' 'just' 'buy']
2 ['dryer' 'dry' 'lint' 'clothes' 'drain' 'drying' 'door' 'great' 'just'
 'don' 'time' 'like']
3 ['years' 'warranty' 'service' 'lg' 'repair' 'buy' 'months' 'year' 'ge'
 'dryer' 'bought' 'machine']
4 ['water' 'machine' 'clothes' 'cycle' 'wash' 'load' 'time' 'use' 'like'
 'laundry' 'dryer' 'just']
5 ['clothes' 'clean' 'water' 'does' 'dryer' 'wash' 'love' 'buy' 'machine'
 'ge' 'come' 'doesn']


In [6]:
d.drop(columns="text_l").to_csv("data_clean/reviews_laundry_features.csv", index=False)